# LLM Inference in Runpod

In [ ]:
# 1. SETUP ENV VARS FIRST (Before imports)
import os
os.environ["HF_HOME"] = "/workspace"
os.environ["HF_HUB_CACHE"] = "/workspace/hub"
# Force stable vLLM engine to avoid "EngineCore" crashes
os.environ["VLLM_USE_V1"] = "0"


In [ ]:
!python -m pip install --upgrade pip -q
!pip install uv -qU

# 1. Force upgrade the conflicting library
!python -m pip install --upgrade typing_extensions

# 2. Fix potential uv conflicts
!uv pip install --system --upgrade typing_extensions

# 3. NOW run your other installs
!uv pip install google.generativeai --system -qU
!uv pip install unsloth tensorboard -qU --system
!uv pip install vllm -qU --system

# 4. RESTART KERNEL logic (if in a notebook)
import sys
if "typing_extensions" in sys.modules:
    # If it was already loaded, we must restart to pick up the new version
    print("Restarting kernel to apply updates...")
    import os
    os._exit(00)


In [ ]:
from huggingface_hub import HfFolder, login

# Check if a token is already saved
if HfFolder.get_token() is None:
    login()  # Will prompt only if not logged in


In [ ]:
import os
os.environ["HF_HOME"] = "/workspace"
os.environ["HF_HUB_CACHE"] = "/workspace/hub" # (recommended) override just the repo cache
print(os.environ["HF_HOME"])


In [ ]:
# Helper to clear cuda without restarting the kernel.

import torch, gc, inspect, sys

def clear_old_model_refs():
    """
    Delete `teacher` `model` and `tokenizer` (if they exist) from the callerÃ¢â‚¬â„¢s
    local *and* global scope, then garbage-collect and free GPU cache.
    """

    #
    frm = inspect.currentframe().f_back
    caller_locals  = frm.f_locals
    caller_globals = frm.f_globals

    for var in ("teacher", "model", "tokenizer"):
        if var in caller_locals:
            try:
                del caller_locals[var]
                if var in sys.modules:   # rarely needed
                    del sys.modules[var]
                print(f"deleted local  {var}")
            except Exception as e:
                print(f"could not delete local {var}: {e}")

        if var in caller_globals:
            try:
                del caller_globals[var]
                print(f"deleted global {var}")
            except Exception as e:
                print(f"could not delete global {var}: {e}")

    # Python & CUDA cleanup 
    torch.cuda.empty_cache()
    print("GPU cache cleared.")



In [ ]:
def formatting_func_inference(batch):
    """
    Convert a batch of inputs into chat-formatted prompts for INFERENCE.
    Only includes System and User roles, and adds the generation prompt.
    """
    out = []
    q_column = 'input'
    
    # We iterate only over the input column (title_and_ingredients)
    for title_and_ingredients in batch[q_column]:

        messages = [
            {
                "role": "system", 
                "content": (
                     "You are a culinary assistant. "
                    "Write step-by-step cooking directions using the given title and ingredients. "
                    "Use all relevant ingredients"
                    "Do NOT repeat the ingredient list. "
                    "Use complete sentences."
                    "Use numbered steps with action verbs."
                )
            },
            {"role": "user", "content": title_and_ingredients},
            # Note: No assistant message here!
        ]
        
        # apply_chat_template handles the structure
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True, # CRITICAL: Appends the assistant header
            enable_thinking=False,       # Recommendation: Disable for direct recipes
        )

        # Qwen/ChatML usually doesn't need the BOS check, 
        # but we keep it if your training pipeline strictly required it.
        bos = tokenizer.bos_token or "<bos>"
        if text.startswith(bos):
            text = text[len(bos):]

        out.append(text)

    return out


In [ ]:
clear_old_model_refs()


Load the Model

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer


# 4. LOAD MODEL & TOKENIZER
#MODEL_PATH = "nijumich/Qwen2.5-7B-Instruct-recipieNLG_V1-1ep-20260220-003813-ft-1gpu"
MODEL_PATH ="Qwen/Qwen2.5-7B-Instruct"

llm = LLM(
    model=MODEL_PATH,
    max_model_len=1024,
    trust_remote_code=True,
    gpu_memory_utilization=0.9, # Good for RTX 2000 Ada / A100. Reduce to 0.7 for 2080/T4.
    enforce_eager=True # Recommended for stability on older/mid-tier cards
)

# Use the tokenizer directly from vLLM to ensure consistency
tokenizer = llm.get_tokenizer()



Load the dataset

In [ ]:
from datasets import load_dataset
offline = True
if offline: 
    dataset = load_dataset("json", data_files={"test": "/workspace/ADVANCED-fine-tuning/fine-tune/datasets/test_10k.jsonl"})
    test_ds = dataset["test"]
    ft_test_data = test_ds #test_ds.select(range(100))  # Take only first 100 rows
else :
    ft_dataset_name = "nijumich/recipieNLG_V1"   
    ft_data = load_dataset(ft_dataset_name)
    ft_train_data = ft_data["train"]
    ft_eval_data = ft_data["validation"]
    ft_test_data = ft_data["test"][:100]  # Take only first 100 rows
print(len(ft_test_data))


Inference Generate model outputs (Qwen3-4B)

In [ ]:
# 7. PREPARE BATCH
# We take the first 10 examples. 
# Slicing [:10] returns a DICT of lists: {'input': [...], 'output': [...]}
# This matches perfectly with 'formatting_func_inference' expecting a batch dict.
small_batch = ft_test_data[:10] 

# Corrected function call name
prompts = formatting_func_inference(small_batch)


In [ ]:
# 8. DEFINE SAMPLING
sampling_params = SamplingParams(
    temperature=0.0,
    top_p=0.95,
    max_tokens=1024,
    stop=["<|im_end|>", "<|endoftext|>"]
)


Generate  output from the model

In [ ]:
# 9. RUN GENERATION
print(f"Processing {len(prompts)} recipes...")
outputs = llm.generate(prompts, sampling_params)

# Optional: Print first result to verify
print(f"\n--- Sample Output ---\n{outputs[0].outputs[0].text}")


In [ ]:
outputs[2].outputs[0].text


### Setup Config and Evaluation Functions

In [ ]:
# -----------------------------
# Core imports and run settings
# -----------------------------
# json: persist manifest/checkpoint summary files.
# os: read optional run-level env overrides.
# re: normalize ingredient text.
# datetime/timezone: generate UTC run id and timestamps.
# Path: file system-safe path joins.
import ast
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
# numpy/pandas: tabular processing + NaN handling for failed metrics.
# tqdm: chunk-level progress bar.
import numpy as np
import pandas as pd
from tqdm import tqdm
# -----------------------------
# HYBRID PIPELINE CONFIG
# -----------------------------
# RUN_ID controls artifact naming and resume behavior.
# If RUN_ID_OVERRIDE is set, you can continue a previous run deterministically.
RUN_ID = os.getenv("RUN_ID_OVERRIDE") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
# Central output locations.
RUNS_DIR = Path("../runs")
RUN_DIR = RUNS_DIR / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(RUN_DIR)
# Canonical output files for this run.
PREDICTIONS_PARQUET_PATH = RUN_DIR / "predictions.parquet"
SAMPLE_CSV_PATH = RUN_DIR / "predictions_sample.csv"
MANIFEST_PATH = RUN_DIR / "run_manifest.json"
CHECKPOINT_PATH = RUN_DIR / "checkpoint.json"
# Runtime controls.
BATCH_SIZE = 32
CHUNK_SIZE = 1024
MAX_SAMPLES = len(ft_test_data)
WRITE_SAMPLE_CSV = True
SAMPLE_CSV_LIMIT = 5000
RESUME = True
START_INDEX = 0
# Known extra columns expected from input data.
# Any additional columns from dataset features are also auto-carried later.
KNOWN_EXTRA_COLUMNS = [
    "ner_ingredients",
    "ingredients_normalized",
    "ingredients_bullets",
    "directions_normalized",
]
# Stable base schema that must exist in all persisted outputs.
BASE_COLUMNS = ["run_id", "sample_id", "input_raw", "input_prompt", "reference", "prediction"]
# Minimal inline metrics intentionally kept cheap and robust.
INLINE_METRIC_COLUMNS = [
    "output_tokens",
    "ingredient_coverage",
    "repetition_ratio",
    "parse_status",
    "error_message",
]
REPO_ROOT = Path("..").resolve()
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from scripts.inline_metrics import compute_inline_metrics, format_ingredient_field

def _to_json_safe(obj):
    """Recursively convert objects to JSON-serializable primitives.
    This avoids manifest-write failures when config objects contain
    tuples or custom types.
    """
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    if isinstance(obj, dict):
        return {k: _to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_json_safe(v) for v in obj]
    return str(obj)
def sampling_config_snapshot(params):
    """Capture a small, stable subset of sampling params for reproducibility.
    We only store keys needed for run comparison and replay.
    """
    keys = ["temperature", "top_p", "max_tokens", "stop"]
    snap = {}
    for k in keys:
        snap[k] = getattr(params, k, None)
    return _to_json_safe(snap)




Inference Loop

In [ ]:
# ----------------------------------------------------------
# Inference loop: generate, score minimal metrics, and persist
# ----------------------------------------------------------
# Build the dynamic schema.
# - Preserve known extra columns.
# - Also preserve any dataset-provided columns except input/output.
all_columns_in_data = list(ft_test_data.features.keys()) if hasattr(ft_test_data, "features") else []
dynamic_extra_columns = [c for c in all_columns_in_data if c not in ["input", "output"]]
extra_columns = []
for c in KNOWN_EXTRA_COLUMNS + dynamic_extra_columns:
    if c not in extra_columns and c not in BASE_COLUMNS and c not in INLINE_METRIC_COLUMNS:
        extra_columns.append(c)
# Final column order is deterministic for easier downstream diffs and joins.
schema_columns = BASE_COLUMNS + extra_columns + INLINE_METRIC_COLUMNS
# Sample CSV bookkeeping.
sample_rows_written = 0
sample_csv_header_written = False
if WRITE_SAMPLE_CSV and SAMPLE_CSV_PATH.exists() and START_INDEX == 0:
    # New full run: remove old sample file to avoid mixing runs.
    SAMPLE_CSV_PATH.unlink()
# Process dataset in chunks for memory safety and resumability.
for chunk_start in tqdm(range(START_INDEX, MAX_SAMPLES, CHUNK_SIZE), desc="Inference chunks"):
    chunk_end = min(chunk_start + CHUNK_SIZE, MAX_SAMPLES)
    batch = ft_test_data[chunk_start:chunk_end]
    # Build prompts from raw batch rows.
    prompts = formatting_func_inference(batch)
    # Generate in mini-batches to fit model/runtime constraints.
    outputs = []
    for i in range(0, len(prompts), BATCH_SIZE):
        outputs.extend(llm.generate(prompts[i:i + BATCH_SIZE], sampling_params))
    # Build row objects for this chunk.
    chunk_rows = []
    for local_idx, (prompt, output) in enumerate(zip(prompts, outputs)):
        sample_id = chunk_start + local_idx
        # Core required fields.
        row = {
            "run_id": RUN_ID,
            "sample_id": sample_id,
            "input_raw": batch["input"][local_idx] if "input" in batch else None,
            "input_prompt": prompt,
            "reference": batch["output"][local_idx] if "output" in batch else None,
            "prediction": output.outputs[0].text if output.outputs else "",
        }
        # Pass-through any extra dataset columns.
        for col in extra_columns:
            if col in batch:
                val = batch[col][local_idx]
                row[col] = format_ingredient_field(val) if col == "ner_ingredients" else val
            else:
                row[col] = None

        # Inline metric compute must never block persistence.
        try:
            row.update(compute_inline_metrics(row))
            row["parse_status"] = "ok"
            row["error_message"] = None
        except Exception as exc:
            row["output_tokens"] = np.nan
            row["ingredient_coverage"] = np.nan
            row["repetition_ratio"] = np.nan
            row["parse_status"] = "error"
            row["error_message"] = str(exc)
        chunk_rows.append(row)
    # Normalize chunk dataframe to expected schema.
    chunk_df = pd.DataFrame(chunk_rows)
    for col in schema_columns:
        if col not in chunk_df.columns:
            chunk_df[col] = None
    chunk_df = chunk_df[schema_columns]
    # Primary storage: one parquet part per chunk.
    part_path = RUN_DIR / f"part_{chunk_start:09d}_{chunk_end - 1:09d}.parquet"
    chunk_df.to_parquet(part_path, index=False)
    # Optional lightweight CSV sample for quick manual inspection.
    if WRITE_SAMPLE_CSV and sample_rows_written < SAMPLE_CSV_LIMIT:
        remaining = SAMPLE_CSV_LIMIT - sample_rows_written
        sample_df = chunk_df.head(remaining)
        sample_df.to_csv(
            SAMPLE_CSV_PATH,
            mode="a",
            index=False,
            header=(not sample_csv_header_written),
            encoding="utf-8",
        )
        sample_rows_written += len(sample_df)
        sample_csv_header_written = True
    # Update checkpoint after each chunk write.
    checkpoint_payload = {
        "run_id": RUN_ID,
        "last_chunk_start": chunk_start,
        "last_chunk_end": chunk_end - 1,
        "last_written_sample_id": chunk_end - 1,
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    CHECKPOINT_PATH.write_text(json.dumps(checkpoint_payload, indent=2), encoding="utf-8")
# Write run manifest once inference persistence is complete.
manifest = {
    "schema_version": "1.0",
    "run_id": RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "paths": {
        "run_dir": str(RUN_DIR),
        "checkpoint": str(CHECKPOINT_PATH),
        "sample_csv": str(SAMPLE_CSV_PATH) if WRITE_SAMPLE_CSV else None,
        "predictions_parquet": str(PREDICTIONS_PARQUET_PATH),
    },
    "config": {
        "batch_size": BATCH_SIZE,
        "chunk_size": CHUNK_SIZE,
        "max_samples": MAX_SAMPLES,
        "start_index": START_INDEX,
        "resume": RESUME,
        "sampling_params": sampling_config_snapshot(sampling_params),
    },
    "columns": schema_columns,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print({
    "parts_written": len(list(RUN_DIR.glob("part_*.parquet"))),
    "manifest": str(MANIFEST_PATH),
    "checkpoint": str(CHECKPOINT_PATH),
})


Conatenate all the parquet files in the output folder into a single parquet file

In [ ]:
# ----------------------------------------------------------
# Load chunk parts and materialize the authoritative parquet
# ----------------------------------------------------------
# Read all chunk files produced by the inference stage.
part_files = sorted(RUN_DIR.glob("part_*.parquet"))
if not part_files:
    raise RuntimeError(f"No parquet parts found in {RUN_DIR}")
# Combine chunk outputs into one dataframe for evaluation/reporting.
df_infer = pd.concat([pd.read_parquet(p) for p in part_files], ignore_index=True)
# Persist single-file authoritative artifact required by the workflow contract.
df_infer.to_parquet(PREDICTIONS_PARQUET_PATH, index=False)
print(f"Loaded {len(df_infer):,} rows from {len(part_files)} parquet parts")
print(f"Authoritative parquet: {PREDICTIONS_PARQUET_PATH}")
df_infer.head(2)
